In [ ]:
#!/usr/bin/env python3
"""
===============================================================================
 preprocess_lowfreq.py  —  re-epoch the FineMI EEG at a LOW frequency band
===============================================================================

EITHER of these works - pick whichever you prefer.

  A) as a file          upload it to /content, then in a cell:
                            !python preprocess_lowfreq.py

  B) pasted into a cell  paste the whole file into one cell and run it.
                         Edit the CONFIG block near the top to change the band
                         or the paths; command-line flags do not exist in this
                         mode and argv is ignored on purpose.

It installs MNE itself if it is missing, so you do not need a pip cell either
way.

ABOUT "You must restart the runtime" / the `decorator` warning
--------------------------------------------------------------
IGNORE IT. Colab imports `decorator` at startup (IPython uses it), so when pip
installs MNE and pulls a newer `decorator`, pip notices the old one is already
in memory and warns. Restarting never fixes it - the same cycle repeats on
every fresh runtime. MNE works fine with the loaded version.

It cannot affect this script at all: `!python preprocess_lowfreq.py` runs in a
SEPARATE process, so nothing loaded in your notebook's memory applies here. The
script prints the MNE version it actually got, so you can see for yourself.

WHAT THIS IS
------------
A standalone version of the EEG half of your preprocessing notebook. The fNIRS
code, the alignment code and the debug cells are all gone. The EEG functions
below are copied VERBATIM from your notebook - same montage, same average
reference, same resampling, same tmin/tmax, same baseline, same trigger
handling - so the only thing that differs from the files you already have is
the band-pass band.

WHY
---
Your .npz epochs are filtered 8-30 Hz. Everything slower than 8 Hz was removed
before any model ever saw the data.

Ofner, Schwarz, Pereira & Muller-Putz (2017, PLOS One), "Upper limb movements
can be decoded from the time-domain of low-frequency EEG", decoded six
movements of the SAME arm (elbow flexion/extension, forearm supination/
pronation, hand open/close) from 0.3-3 Hz and got 55% for executed movement and
27% for MOTOR IMAGERY, against 16.7% chance. Your task is imagery, so 27% is
the number to hold in mind, not 55%. They also report that mu and beta contain
LESS information about movement kinematics than the low-frequency band, and
recommend combining the two.

This script produces the second half of that combination.

OUTPUT
------
A NEW folder. Nothing you already have is touched or overwritten:

    .../extracted/FineMI_0.3_3hz/
        subject1/  subject2/  ... subject18/          (empty, the loader needs them)
        subject1_eeg_epochs_0.3_3hz_250hz.npz
        ...

WHY A SEPARATE FOLDER, NOT A SECOND FILE ALONGSIDE THE OLD ONE
--------------------------------------------------------------
Your training code loads data with

    cand = glob.glob(f"subject{sid}_eeg_epochs_*.npz");  np.load(cand[0])

That glob is UNSORTED and takes [0]. Drop a second .npz next to the first and
it silently picks whichever the filesystem happens to return - possibly the
8-30 file for one subject and the 0.3-3 file for the next, with no error and no
warning. A separate folder makes switching a one-line change instead.

AFTER IT FINISHES
-----------------
Point ONE line of the 8-class notebook at the new folder:

    DATASET_ROOT = '.../extracted/FineMI_0.3_3hz'

and run the 'trunk_merge' arm. NOT 'fused' - see the note at the bottom.
===============================================================================
"""
import os
import glob
import re
import gc
import argparse

import subprocess
import sys

import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# --------------------------------------------------------------------------
# MNE bootstrap - no pip cell needed, and no restart needed either.
# --------------------------------------------------------------------------
try:
    import mne
except ImportError:
    print("MNE not found, installing (about 30 s)...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne"],
                   check=True)
    import mne

mne.set_log_level('ERROR')
print(f"MNE {mne.__version__} loaded OK  (python {sys.version.split()[0]})")


# ==========================================================================
#  CONFIG — these are used when you PASTE THIS INTO A NOTEBOOK CELL.
#  (Running it as `!python preprocess_lowfreq.py` uses the same values as
#   defaults, and command-line flags override them.)
# ==========================================================================
# 0.5 Hz, NOT 0.3. The dataset paper (Yi et al., Sci. Data 2025) states the
# acquisition band-pass was 0.5-100 Hz, so there is literally nothing below
# 0.5 Hz in the .cnt files - asking for 0.3 just widens the filter transition
# for no signal. Ofner's published band is 0.3-3 Hz; 0.5-3 is the closest this
# recording can give.
L_FREQ = 0.5          # low cutoff Hz   — your existing files used 8.0
H_FREQ = 3.0          # high cutoff Hz  — your existing files used 30.0
#
# Worth knowing: the dataset paper's OWN baseline used 4-40 Hz, not 8-30. To
# reproduce their preprocessing instead, set L_FREQ = 4.0, H_FREQ = 40.0.

SRC = '/content/drive/MyDrive/FineMI2'   # folder with subject1..18/EEG/*.cnt
OUT = '/content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly'            # None -> a sibling of SRC named FineMI_<band>hz.
                      # On Colab this MUST end up under /content/drive/MyDrive/ —
                      # the Drive mount root itself is not writable. Set an
                      # explicit path here if the automatic one is refused.
SUBJECTS = ''         # '' = all; or e.g. '1,2,3' to test on a few first
FORCE = False         # True = recompute subjects already written
# ==========================================================================


def _running_in_notebook():
    """True when this code was pasted into a Jupyter/Colab cell.

    In that case sys.argv belongs to the kernel, not to us
    (`colab_kernel_launcher.py -f /root/.local/.../kernel-xxxx.json`), and
    handing it to argparse makes it exit with 'unrecognized arguments'. So we
    ignore argv entirely and use the CONFIG block above.
    """
    if 'ipykernel' in sys.modules or 'google.colab' in sys.modules:
        return True
    return 'kernel_launcher' in (sys.argv[0] if sys.argv else '')

"""
EEG preprocessing + epoch extraction for Multi-Joint Upper Limb MI Dataset (.cnt)

- Loads all CNT blocks for a subject (handles Subject1 special naming)
- Marks known non-EEG channels as misc
- Sets montage (standard_1020, ignore missing)
- Sets average reference (applied immediately; projection=False)
- Bandpass filters (default 8–30 Hz) on EEG picks
- Resamples to target sfreq (default 250 Hz)
- Extracts events via annotations first; if none, tries stim channel fallback
- Creates epochs aligned to triggers 1–8
- Creates labels 0–7 correctly (based on trigger numbers 1–8), NOT on MNE arbitrary codes

Works as a script or importable module.

NOTE:
- Do NOT put "!pip install ..." inside a .py module. Install mne once in your environment.
"""



# Map trigger numbers (1-8) to MI task names
MI_TASKS = {
    1: "hand_open_close",
    2: "wrist_flexion_extension",
    3: "wrist_abduction_adduction",
    4: "elbow_pronation_supination",
    5: "elbow_flexion_extension",
    6: "shoulder_pronation_supination",
    7: "shoulder_abduction_adduction",
    8: "shoulder_flexion_extension",
}

DEFAULT_L_FREQ = 8.0
DEFAULT_H_FREQ = 30.0
DEFAULT_TARGET_SFREQ = 250.0


# =============================================================================
# Helper utilities
# =============================================================================

def _sort_cnt_files(cnt_files, is_subject1: bool):
    """Sort CNT files in correct order, handling Subject1 special naming."""
    if is_subject1:
        def get_block_number(filename):
            name = os.path.basename(filename).lower()
            if "block1-4" in name:
                return 1
            for b in [5, 6, 7, 8]:
                if f"block{b}" in name:
                    return b
            match = re.search(r"block(\d+)", name)
            return int(match.group(1)) if match else 999
        return sorted(cnt_files, key=get_block_number)
    else:
        def get_block_number(filename):
            name = os.path.basename(filename).lower()
            match = re.search(r"block(\d+)", name)
            return int(match.group(1)) if match else 999
        return sorted(cnt_files, key=get_block_number)


def _safe_print_annotations(raw, n=10):
    """Print a small annotation preview safely."""
    if hasattr(raw, "annotations") and raw.annotations is not None and len(raw.annotations) > 0:
        print("Annotation descriptions (first):", raw.annotations.description[:n])
        print("Annotation onsets (first):      ", np.round(raw.annotations.onset[:n], 3))
    else:
        print("No annotations present in raw.")


def _get_stim_channels(raw):
    """Return list of stim channels (type 'stim') in raw."""
    stim_picks = mne.pick_types(raw.info, meg=False, eeg=False, stim=True, eog=False, misc=False)
    return [raw.ch_names[i] for i in stim_picks]


def extract_trigger_num(event_name) -> int | None:
    """
    Extract trigger number 1..8 from an event name/description string.
    Examples: '1', 'S  1', 'Stimulus/3', 'trigger_5'
    """
    s = str(event_name).strip()
    if s.isdigit():
        num = int(s)
        return num if 1 <= num <= 8 else None
    match = re.search(r"(\d+)", s)
    if match:
        num = int(match.group(1))
        return num if 1 <= num <= 8 else None
    return None


def _events_from_annotations(raw):
    """Try extracting events from annotations. Returns (events, event_dict) or (None, None)."""
    try:
        events, event_dict = mne.events_from_annotations(raw, verbose=False)
        return events, event_dict
    except Exception:
        return None, None


def _events_from_stim(raw, stim_channel=None):
    """
    Try extracting events from stim channel using mne.find_events.
    Returns (events, event_dict, stim_channel_used) or (None, None, None)
    """
    stim_channels = _get_stim_channels(raw)
    if len(stim_channels) == 0:
        return None, None, None

    stim_channel_used = stim_channel if stim_channel is not None else stim_channels[0]
    if stim_channel_used not in stim_channels:
        return None, None, None

    try:
        events = mne.find_events(raw, stim_channel=stim_channel_used, verbose=False)
        if events is None or len(events) == 0:
            return None, None, stim_channel_used
        # event_dict from codes (note: codes may not be 1-8!)
        unique_codes = np.unique(events[:, -1])
        event_dict = {str(int(code)): int(code) for code in unique_codes}
        return events, event_dict, stim_channel_used
    except Exception:
        return None, None, stim_channel_used


def get_events_dual_path(raw, verbose=True):
    """
    Dual-path event extraction:
    1) annotations -> mne.events_from_annotations
    2) stim channel -> mne.find_events
    Returns dict with method, events, event_dict, stim_channel_used.
    """
    result = {
        "method": None,
        "events": None,
        "event_dict": None,
        "stim_channel_used": None,
    }

    # A) annotations
    events, event_dict = _events_from_annotations(raw)
    if events is not None and len(events) > 0 and event_dict is not None and len(event_dict) > 0:
        result.update({"method": "annotations", "events": events, "event_dict": event_dict})
        if verbose:
            print(f"✓ Events from annotations: {len(events)} events, {len(event_dict)} types")
        return result

    if verbose:
        print("⚠️ Annotation-based event extraction failed or returned zero events. Trying stim channel...")

    # B) stim
    events, event_dict, stim_used = _events_from_stim(raw, stim_channel=None)
    if events is not None and len(events) > 0 and event_dict is not None and len(event_dict) > 0:
        result.update({"method": "stim", "events": events, "event_dict": event_dict, "stim_channel_used": stim_used})
        if verbose:
            print(f"✓ Events from stim channel '{stim_used}': {len(events)} events, {len(event_dict)} codes")
        return result

    # failure
    if verbose:
        stim_channels = _get_stim_channels(raw)
        print("❌ No events found via annotations or stim.")
        print(f"   Stim channels detected: {stim_channels if stim_channels else 'None'}")
    return result


def build_epoch_mappings(event_dict, verbose=True):
    """
    Build:
      - event_id: {event_name: event_code} for triggers 1..8
      - code_to_trigger: {event_code: trigger_num} for triggers 1..8
    IMPORTANT:
      - event_id must use *codes* from event_dict (for MNE epoching)
      - code_to_trigger maps codes to trigger numbers (for label creation)
      - Labels are derived from trigger numbers (1..8 -> 0..7), NOT from codes
    """
    event_id = {}
    code_to_trigger = {}

    for event_name, code in event_dict.items():
        trig = extract_trigger_num(event_name)
        if trig is not None and 1 <= trig <= 8:
            event_id[event_name] = code
            code_to_trigger[int(code)] = int(trig)

    if verbose:
        print(f"Event types matching triggers 1–8: {len(event_id)}")
        if len(event_id) > 0:
            # show mapping preview
            for k in sorted(event_id.keys(), key=lambda x: (extract_trigger_num(x) or 999, str(x))):
                print(f"  '{k}' -> code {event_id[k]} (trigger {extract_trigger_num(k)})")

    return event_id, code_to_trigger


# =============================================================================
# Core preprocessing functions
# =============================================================================

def load_subject_eeg(
    subject_path,
    target_sampling_rate=DEFAULT_TARGET_SFREQ,
    l_freq=DEFAULT_L_FREQ,
    h_freq=DEFAULT_H_FREQ,
    verbose=True
):
    """
    Load and preprocess EEG data from CNT files for one subject:
    - loads all blocks
    - concatenates
    - sets montage (standard_1020, ignore missing)
    - average reference (applied)
    - bandpass filter on EEG channels
    - resample
    """
    subject_name = os.path.basename(subject_path.rstrip(os.sep))
    is_subject1 = subject_name.lower() == "subject1"

    # ---- find the .cnt files -------------------------------------------
    # The original notebook looked ONLY in <subject>/EEG/*.cnt, one level
    # deep, case-sensitive. That misses layouts where the blocks sit in
    # <subject>/EEG/block1/x.cnt, or the folder is named "eeg", or the
    # extension is ".Cnt". Search the strict location first (so a normal
    # layout behaves exactly as before), then fall back to a recursive,
    # case-insensitive walk of the whole subject folder.
    eeg_folder = os.path.join(subject_path, "EEG")
    cnt_files = (glob.glob(os.path.join(eeg_folder, "*.cnt")) +
                 glob.glob(os.path.join(eeg_folder, "*.CNT")))

    if not cnt_files:
        cnt_files = [os.path.join(r, f)
                     for r, _, fs in os.walk(subject_path)
                     for f in fs if f.lower().endswith(".cnt")]
        if cnt_files and verbose:
            where = sorted({os.path.relpath(os.path.dirname(f), subject_path)
                            for f in cnt_files})
            print(f"  (not in EEG/ — found {len(cnt_files)} .cnt files in: "
                  f"{', '.join(where)})")

    if not cnt_files:
        raise FileNotFoundError(
            f"No .cnt files anywhere under {subject_path} "
            f"(searched recursively, any capitalisation)")

    cnt_files = _sort_cnt_files(cnt_files, is_subject1=is_subject1)

    print(f"  {subject_name}: {len(cnt_files)} .cnt block(s)"
          + ("   <-- FEWER THAN EXPECTED, check the folder" if len(cnt_files) < 4 else ""))
    if verbose:
        print(f"Found {len(cnt_files)} EEG blocks for {subject_name}")
        print("Loading blocks in order:")
        for i, f in enumerate(cnt_files, 1):
            print(f"  {i}. {os.path.basename(f)}")

    all_blocks = []
    for cnt_file in cnt_files:
        if verbose:
            print(f"  Loading {os.path.basename(cnt_file)}...")
        try:
            block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
        except Exception as e:
            raise RuntimeError(f"Failed to load {cnt_file}: {e}")

        # Mark known non-EEG channels as misc (adjust if needed)
        misc_channels = ["CB1", "CB2", "HEO", "VEO", "EKG", "EMG"]
        present_misc = [ch for ch in misc_channels if ch in block.ch_names]
        if present_misc:
            block.set_channel_types({ch: "misc" for ch in present_misc}, verbose=False)

        # Montage (safe)
        try:
            block.set_montage("standard_1020", on_missing="ignore", verbose=False)
        except Exception:
            if verbose:
                print("    Warning: Could not set montage (standard_1020). Continuing...")

        all_blocks.append(block)

    if len(all_blocks) == 0:
        raise ValueError("No blocks loaded successfully.")

    # Ensure sfreq consistent
    sfreqs = {blk.info["sfreq"] for blk in all_blocks}
    if len(sfreqs) > 1:
        raise RuntimeError(f"Blocks have different sampling rates: {sfreqs}")

    raw = all_blocks[0] if len(all_blocks) == 1 else mne.concatenate_raws(all_blocks, preload=True, verbose=False)

    # Apply average reference BEFORE filtering
    try:
        raw.set_eeg_reference("average", projection=False, verbose=False)
    except Exception as e:
        if verbose:
            print(f"Warning: Could not set average reference: {e}")

    # Filter only EEG channels
    eeg_picks = mne.pick_types(raw.info, meg=False, eeg=True, eog=False, stim=False, misc=False)
    if len(eeg_picks) == 0:
        raise ValueError("No EEG channels found (check channel types).")

    if verbose:
        print(f"Applying bandpass filter ({l_freq}-{h_freq} Hz) on EEG channels...")
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks=eeg_picks, verbose=False)

    # Resample
    current_rate = float(raw.info["sfreq"])
    if abs(current_rate - float(target_sampling_rate)) > 0.1:
        if verbose:
            print(f"Resampling from {current_rate:.2f} Hz to {target_sampling_rate:.2f} Hz...")
        raw.resample(target_sampling_rate, verbose=False)

    if verbose:
        n_total = len(raw.ch_names)
        n_eeg = len(mne.pick_types(raw.info, meg=False, eeg=True, eog=False, stim=False, misc=False))
        duration = raw.times[-1] if len(raw.times) > 0 else 0.0
        print("\n✓ Loaded successfully!")
        print(f"  EEG channels: {n_eeg} / Total channels: {n_total}")
        print(f"  Sampling rate: {raw.info['sfreq']:.2f} Hz")
        print(f"  Duration: {duration:.1f} s")
        _safe_print_annotations(raw, n=10)
        stim_channels = _get_stim_channels(raw)
        print("Stim channels detected:", stim_channels if stim_channels else "None")

    return raw


def extract_epochs(
    raw,
    start_time=-0.5,
    end_time=4.0,
    baseline=None,
    verbose=True,
    stim_channel=None,
    code_remapping=None,
):
    """
    Extract epochs and labels from continuous raw EEG.

    - Finds events using annotations first; falls back to stim.
    - Uses event NAMES to infer trigger numbers 1..8.
    - Uses event CODES only for epoching (as required by MNE).
    - Produces labels 0..7 based on trigger numbers (1..8).

    Parameters
    ----------
    code_remapping : dict or None
        Optional explicit mapping {event_code: trigger_num(1..8)}.
        Use this if stim channel codes are not directly tied to trigger labels.
        Example: {10001: 1, 10002: 2, ...} if your codes are 10001-10008.
        If provided, it overrides automatic mapping from event names.
    """
    if start_time >= end_time:
        raise ValueError(f"start_time ({start_time}) must be < end_time ({end_time})")
    if len(raw.times) == 0:
        raise ValueError("Raw data has no time points.")

    # Dual path: annotations then stim (optionally choose stim_channel)
    events, event_dict = _events_from_annotations(raw)
    method = "annotations"
    stim_used = None

    if events is None or event_dict is None or len(events) == 0 or len(event_dict) == 0:
        method = "stim"
        events, event_dict, stim_used = _events_from_stim(raw, stim_channel=stim_channel)

    if events is None or event_dict is None or len(events) == 0 or len(event_dict) == 0:
        stim_channels = _get_stim_channels(raw)
        raise ValueError(
            "No events found via annotations or stim channel.\n"
            "Tips:\n"
            "  - Check raw.annotations (should contain event descriptions)\n"
            "  - Check stim channels: " + (f"{stim_channels}" if stim_channels else "None found") + "\n"
            "  - Inspect raw data: raw.plot()\n"
            "  - If using stim channel with non-1-8 codes, provide code_remapping parameter"
        )

    if verbose:
        print(f"\nEvent extraction method: {method}")
        if stim_used is not None:
            print(f"Stim channel used: '{stim_used}'")
        print(f"Total events found: {len(events)}")
        # Show a small dictionary preview
        preview_keys = list(event_dict.keys())[:20]
        print(f"Event_dict keys (first {len(preview_keys)}): {preview_keys}")

    # Build event_id and code_to_trigger (from names)
    event_id, code_to_trigger_auto = build_epoch_mappings(event_dict, verbose=verbose)

    if len(event_id) == 0:
        # If stim method returns event_dict with numeric keys like '1','2',... it will work.
        # If keys are not parseable into 1..8 triggers, you need a remapping.
        found_codes = sorted(np.unique(events[:, -1]).tolist()) if events is not None else []
        raise ValueError(
            "Could not identify any triggers 1–8 from event names.\n"
            f"Event names found: {sorted(event_dict.keys())}\n"
            f"Event codes found: {found_codes}\n"
            "If using stim channel with arbitrary codes (not 1-8), you need code_remapping.\n"
            "Example: code_remapping={100: 1, 200: 2, 300: 3, ...}"
        )

    # Baseline
    if baseline is None:
        baseline = (start_time, 0.0) if start_time < 0 else (None, 0.0)

    if verbose:
        print(f"Baseline: {baseline}")
        print(f"Epoch window: tmin={start_time}, tmax={end_time}")

    # Epochs (event_id must use *codes*)
    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=start_time,
        tmax=end_time,
        baseline=baseline,
        picks="eeg",
        reject_by_annotation=True,
        preload=True,
        verbose=False,
    )

    if len(epochs) == 0:
        raise ValueError("No epochs were created. Check event timing and tmin/tmax range.")

    # Map epoch event codes -> trigger numbers -> labels
    epoch_codes = epochs.events[:, -1].astype(int)

    if code_remapping is not None:
        code_to_trigger = {int(k): int(v) for k, v in code_remapping.items()}
        if verbose:
            print("\nUsing user-provided code_remapping (code -> trigger_num):")
            for k in sorted(code_to_trigger.keys()):
                print(f"  {k} -> {code_to_trigger[k]}")
    else:
        code_to_trigger = code_to_trigger_auto
        if verbose:
            print("\nUsing automatic code->trigger mapping from event_dict names:")
            for k in sorted(code_to_trigger.keys()):
                print(f"  {k} -> {code_to_trigger[k]}")

    trigger_nums = np.array([code_to_trigger.get(c, -1) for c in epoch_codes], dtype=int)
    if np.any(trigger_nums == -1):
        bad = sorted(set(epoch_codes[trigger_nums == -1].tolist()))
        available = sorted(code_to_trigger.keys())
        raise ValueError(
            f"Some epoch event codes could not be mapped to triggers 1–8: {bad}\n"
            f"Available mappings: {available}\n"
            "If you used stim channel and codes are arbitrary, pass code_remapping parameter."
        )

    labels = trigger_nums - 1
    if labels.min() < 0 or labels.max() > 7:
        raise ValueError(f"Invalid label range: {labels.min()}..{labels.max()} (expected 0..7).")

    if verbose:
        print("\n✓ Extracted epochs")
        print(f"  Trials: {len(epochs)}")
        print(f"  Data shape: {epochs.get_data().shape} = (trials, channels, timepoints)")
        print(f"  Unique triggers: {sorted(np.unique(trigger_nums).tolist())}")
        # label distribution
        unique_labels, counts = np.unique(labels, return_counts=True)
        print("  Label distribution:")
        for lab, cnt in zip(unique_labels, counts):
            trig = int(lab) + 1
            print(f"    Label {lab} (trigger {trig}, {MI_TASKS.get(trig, 'unknown')}): {cnt}")

    return epochs, labels


def labels_to_task_names(labels):
    """Convert integer labels (0–7) to MI task name strings."""
    labels = np.asarray(labels, dtype=int)
    if labels.min() < 0 or labels.max() > 7:
        raise ValueError(f"Labels must be 0..7, got {labels.min()}..{labels.max()}")
    return np.array([MI_TASKS[int(l) + 1] for l in labels])


def epochs_to_numpy(epochs):
    """Convert MNE Epochs to NumPy array."""
    return epochs.get_data()


def save_epochs_numpy(epochs, labels, out_path):
    """Save epochs and labels to compressed NumPy file."""
    data = epochs.get_data()
    labels = np.asarray(labels, dtype=int)
    trial_onsets = epochs.events[:, 0].astype(int)  # sample index in (resampled) raw
    np.savez_compressed(out_path, data=data, labels=labels, onsets=trial_onsets)
    print(f"Saved: {out_path}")
    print(f"  data shape: {data.shape}")
    print(f"  labels shape: {labels.shape}")
    print(f"  onsets shape: {trial_onsets.shape}")

# =============================================================================
#  MAIN
# =============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--l_freq', type=float, default=L_FREQ,
                    help='low cutoff Hz (your existing files used 8.0)')
    ap.add_argument('--h_freq', type=float, default=H_FREQ,
                    help='high cutoff Hz (your existing files used 30.0)')
    ap.add_argument('--expect_trials', type=int, default=320,
                    help='trials per subject the dataset should have (Yi et al.: '
                         '8 blocks x 40 = 320). Warns if a subject comes up short.')
    ap.add_argument('--src', default=SRC,
                    help='folder holding subject1..18/ with their EEG/*.cnt')
    ap.add_argument('--out', default=OUT,
                    help='output folder (default: ../FineMI_<band>hz)')
    ap.add_argument('--subjects', default=SUBJECTS,
                    help='comma-separated subject numbers (default: all)')
    ap.add_argument('--force', action='store_true', default=FORCE,
                    help='recompute subjects whose output already exists')

    # In a notebook, sys.argv is the KERNEL's argv - parsing it kills the cell
    # with 'unrecognized arguments: -f .../kernel-xxxx.json'. Use CONFIG then.
    if _running_in_notebook():
        print("(running inside a notebook - using the CONFIG block at the top,"
              " not command-line flags)")
        a = ap.parse_args([])
    else:
        a = ap.parse_args()

    band_tag = f"{a.l_freq:g}_{a.h_freq:g}hz"

    # Default: a sibling of the source folder. Climbing two levels up landed on
    # /content/drive, the Drive MOUNT ROOT, which is not writable - mkdir there
    # fails with "[Errno 95] Operation not supported". Only /content/drive/MyDrive
    # and below accept writes.
    out_root = a.out or os.path.join(os.path.dirname(os.path.abspath(a.src)),
                                     f"FineMI_{band_tag}")

    # Guard: never try to write straight into the Drive mount root.
    _bad_roots = ('/content/drive', '/content/drive/Shareddrives', '/content/drive/.shortcut-targets-by-id')
    if os.path.dirname(os.path.normpath(out_root)) in _bad_roots:
        out_root = os.path.join('/content/drive/MyDrive', f"FineMI_{band_tag}")
        print(f"NOTE: the derived path sat in the Drive mount root, which is read-only.")
        print(f"      Using {out_root} instead.")

    # Fail now, with a fix, rather than after an hour of filtering.
    try:
        os.makedirs(out_root, exist_ok=True)
        _probe = os.path.join(out_root, '.write_test')
        with open(_probe, 'w') as _f:
            _f.write('ok')
        os.remove(_probe)
    except OSError as e:
        print("\n" + "=" * 74)
        print(f"CANNOT WRITE TO: {out_root}")
        print(f"  {type(e).__name__}: {e}")
        print("=" * 74)
        print("On Colab, Google Drive is only writable under /content/drive/MyDrive/.")
        print("Set OUT in the CONFIG block near the top of this file to a path there,")
        print("for example:")
        print("    OUT = '/content/drive/MyDrive/multi_joint_mi_dataset/extracted"
              f"/FineMI_{band_tag}'")
        print("Then re-run. Nothing has been written yet.")
        return

    def subject_key(p):
        m = re.search(r"subject(\d+)", os.path.basename(p).lower())
        return int(m.group(1)) if m else 9999

    def _find_subject_dirs(root):
        d = sorted([p for p in glob.glob(os.path.join(root, "subject*"))
                    if os.path.isdir(p)], key=subject_key)
        if d:
            return root, d
        # the dataset is often nested one or two levels down (e.g. .../FineMI/FineMI)
        for depth in (1, 2):
            pat = os.path.join(root, *(['*'] * depth), "subject*")
            d = sorted([p for p in glob.glob(pat) if os.path.isdir(p)], key=subject_key)
            if d:
                found_root = os.path.dirname(d[0])
                print(f"NOTE: no subject* folders directly in {root}")
                print(f"      found them in {found_root} — using that as the source.")
                return found_root, d
        return root, []

    a.src, subs = _find_subject_dirs(a.src)
    if a.subjects.strip():
        want = {int(s) for s in a.subjects.split(',') if s.strip()}
        subs = [p for p in subs if subject_key(p) in want]

    print("=" * 74)
    print(f"band      : {a.l_freq} - {a.h_freq} Hz     (your existing files: 8 - 30 Hz)")
    print(f"source    : {a.src}")
    print(f"output    : {out_root}")
    print(f"subjects  : {len(subs)}")
    print("=" * 74)
    if not subs:
        print("No subject folders found - check --src"); return

    done, failed = 0, []
    for p in subs:
        name = os.path.basename(p)
        out_file = os.path.join(out_root, f"{name}_eeg_epochs_{band_tag}_250hz.npz")
        if os.path.exists(out_file) and not a.force:
            print(f"{name:<12} already done, skipping")
            done += 1
            continue
        print(f"\n=== {name} ===", flush=True)
        raw = epochs = labels = None
        try:
            # ONLY l_freq / h_freq differ from the run that made your 8-30 files
            raw = load_subject_eeg(p, l_freq=a.l_freq, h_freq=a.h_freq, verbose=False)
            epochs, labels = extract_epochs(raw, start_time=-4.0, end_time=4.0, baseline=(-4.0, -2.0), verbose=False)
            epochs.crop(tmin=0.0, tmax=4.0 - 1/250)
            os.makedirs(os.path.join(out_root, name), exist_ok=True)  # loader needs it
            n_tr = len(labels)
            if n_tr < a.expect_trials:
                print(f"  !! {n_tr} trials, expected {a.expect_trials} "
                      f"({a.expect_trials - n_tr} missing) — some .cnt blocks were "
                      f"not found or not read. Do NOT use this subject.")
            save_epochs_numpy(epochs, labels, out_file)
            done += 1
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            failed.append(name)
        finally:
            del raw, epochs, labels
            gc.collect()

    print(f"\n{'=' * 74}\n{done} written, {len(failed)} failed {failed}")

    # ---------------------------------------------------------------------
    #  VERIFY against the existing 8-30 Hz files. If the trials or their order
    #  differ, every comparison you make afterwards is meaningless.
    # ---------------------------------------------------------------------
    print("\n" + "=" * 74)
    print("VERIFICATION vs your existing files")
    print("=" * 74)
    print(f"{'subject':<12}{'old shape':>18}{'new shape':>18}{'labels':>10}{'corr':>8}")
    problems = []
    for p in subs:
        name = os.path.basename(p); sid = subject_key(p)
        old = glob.glob(os.path.join(a.src, f"subject{sid}_eeg_epochs_*.npz"))
        new = os.path.join(out_root, f"{name}_eeg_epochs_{band_tag}_250hz.npz")
        if not old or not os.path.exists(new):
            print(f"{name:<12}  a file is missing"); problems.append(name); continue
        o = np.load(old[0], allow_pickle=True); n = np.load(new, allow_pickle=True)
        shape_ok = o['data'].shape == n['data'].shape
        lab_ok = (o['labels'].shape == n['labels'].shape and
                  (np.asarray(o['labels']) == np.asarray(n['labels'])).all())
        # a genuinely different band must NOT look like the old one
        c = float(np.corrcoef(o['data'][:20].ravel(), n['data'][:20].ravel())[0, 1])
        bad = not (shape_ok and lab_ok and abs(c) < 0.5)
        print(f"{name:<12}{str(o['data'].shape):>18}{str(n['data'].shape):>18}"
              f"{('OK' if lab_ok else 'MISMATCH'):>10}{c:>8.3f}"
              + ("   <-- CHECK" if bad else ""))
        if bad:
            problems.append(name)

    print()
    if problems:
        print(f"PROBLEM with: {problems}")
        print("  shape differs   -> different trial counts; the comparison is broken")
        print("  labels MISMATCH -> trials are not in the same order; stop here")
        print("  |corr| > 0.5    -> the new file still looks like 8-30 Hz, i.e. the")
        print("                     band arguments did not take effect")
    else:
        print("ALL GOOD - same trials, same labels, genuinely different frequencies.")
        print("\nNEXT: in EIGHT_CLASS.ipynb change one line,")
        print(f"    DATASET_ROOT = '{out_root}'")
        print("and run the trunk_merge arm only:")
        print("    ARMS = [('trunk_merge', ['--arm','trunk','--val_mode','merge'])]")
        print("\nDo NOT run the 'fused' arm on this data without also setting")
        print(f"    DATA_BAND = ({a.l_freq}, {a.h_freq})   and   N_FB_BANDS = 3")
        print("in run_8class.py. The filter bank is currently hard-coded to split")
        print("8-30 Hz into 6 bands; on 0.3-3 Hz data all six would sit outside the")
        print("passband and carry nothing but filter roll-off.")


if __name__ == "__main__":
    main()


Mounted at /content/drive
MNE not found, installing (about 30 s)...
MNE 1.13.2 loaded OK  (python 3.13.15)
(running inside a notebook - using the CONFIG block at the top, not command-line flags)
band      : 0.5 - 3.0 Hz     (your existing files: 8 - 30 Hz)
source    : /content/drive/MyDrive/FineMI2
output    : /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly
subjects  : 18

=== subject1 ===
  subject1: 5 .cnt block(s)


/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in present_misc}, verbose=False)
/tmp/ipykernel_10261/539804712.py:424: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  block.set_montage("standard_1020", on_missing="ignore", verbose=False)
/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in pre

Saved: /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly/subject1_eeg_epochs_0.5_3hz_250hz.npz
  data shape: (360, 62, 1000)
  labels shape: (360,)
  onsets shape: (360,)

=== subject2 ===
  subject2: 8 .cnt block(s)


/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in present_misc}, verbose=False)
/tmp/ipykernel_10261/539804712.py:424: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  block.set_montage("standard_1020", on_missing="ignore", verbose=False)
/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in pre

Saved: /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly/subject2_eeg_epochs_0.5_3hz_250hz.npz
  data shape: (320, 62, 1000)
  labels shape: (320,)
  onsets shape: (320,)

=== subject3 ===
  subject3: 8 .cnt block(s)


...

Saved: /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly/subject16_eeg_epochs_0.5_3hz_250hz.npz
  data shape: (320, 62, 1000)
  labels shape: (320,)
  onsets shape: (320,)

=== subject17 ===
  subject17: 8 .cnt block(s)


/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in present_misc}, verbose=False)
/tmp/ipykernel_10261/539804712.py:424: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  block.set_montage("standard_1020", on_missing="ignore", verbose=False)
/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in pre

Saved: /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly/subject17_eeg_epochs_0.5_3hz_250hz.npz
  data shape: (320, 62, 1000)
  labels shape: (320,)
  onsets shape: (320,)

=== subject18 ===
  subject18: 8 .cnt block(s)


/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in present_misc}, verbose=False)
/tmp/ipykernel_10261/539804712.py:424: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  block.set_montage("standard_1020", on_missing="ignore", verbose=False)
/tmp/ipykernel_10261/539804712.py:412: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  block = mne.io.read_raw_cnt(cnt_file, preload=True, verbose=False)
/tmp/ipykernel_10261/539804712.py:420: RuntimeWarning: The unit for channel(s) CB1, CB2, EKG, EMG, HEO, VEO has changed from V to NA.
  block.set_channel_types({ch: "misc" for ch in pre

Saved: /content/drive/MyDrive/fixationcrossbaseline_0.5_3hz_MIonly/subject18_eeg_epochs_0.5_3hz_250hz.npz
  data shape: (320, 62, 1000)
  labels shape: (320,)
  onsets shape: (320,)

18 written, 0 failed []

VERIFICATION vs your existing files
subject              old shape         new shape    labels    corr
subject1      a file is missing
subject2      a file is missing
subject3      a file is missing
subject4      a file is missing
subject5      a file is missing
subject6      a file is missing
subject7      a file is missing
subject8      a file is missing
subject9      a file is missing
subject10     a file is missing
subject11     a file is missing
subject12     a file is missing
subject13     a file is missing
subject14     a file is missing
subject15     a file is missing
subject16     a file is missing
subject17     a file is missing
subject18     a file is missing

PROBLEM with: ['subject1', 'subject2', 'subject3', 'subject4', 'subject5', 'subject6', 'subject7', 'subject8', '